In [45]:
import requests #Pour faire des requêtes HTTP
from bs4 import BeautifulSoup #Pour extraire et manipuler des données HTML
import pandas as pd #Manipulation de données

In [106]:
categories_generiques = {"Accueil", "Femmes", "Hommes", "Enfants", "Vêtements"}

variables=["url", "etat", "matiere", "couleur", "date", "categorie", "pays", "likes", "prix", "prix_total"]

df_vinted = pd.DataFrame(columns=variables)
df_vinted.head()

,url,etat,matiere,couleur,date,categorie,pays,likes,prix,prix_total


In [47]:
url="https://www.vinted.fr/catalog?catalog[]=1904&brand_ids[]=12&page=1&time=1778184660"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')


Requête réussie; Code: 200


In [ ]:
urls = [link.get('href') for link in soup.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 
print (urls[0])
print(len(urls))

https://www.vinted.fr/items/8851956521-jeans-zara?referrer=catalog
96


In [49]:
url=urls[0]
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [50]:
#scrapper l'état'
get_etat=soup.select_one('[itemprop="status"]')
etat = get_etat.get_text(" ", strip=True) if get_etat else None

etat

'Très bon état'

In [64]:
#scrapper la matière
get_matiere=soup.select_one('[itemprop="material"]')
if get_matiere:
    matiere = get_matiere.get_text(" ", strip=True)

matiere

In [52]:
#scrapper la couleur
get_couleur=soup.select_one('[itemprop="color"]')
couleur = get_couleur.get_text(" ", strip=True) if get_couleur else None

couleur

'Bleu clair'

In [53]:
#scrapper date
get_date=soup.select_one('[itemprop="upload_date"]')
date = get_date.get_text(" ", strip=True) if get_date else None

date

'Il y a 3 heures'

In [54]:
#scrapper le prix
get_prix = soup.select_one('[data-testid="item-price"]')
get_prix_total = soup.select_one('[data-testid="total-combined-price"]')

prix = get_prix.get_text(" ", strip=True) if get_prix else None
prix_total=get_prix_total.get_text(" ", strip=True) if get_prix else None

print(prix, prix_total)

8,00 € 9,10 €


In [55]:
#scrapper catégories

categories = [
    tag.get_text(" ", strip=True)
    for tag in soup.select('ul.breadcrumbs span[itemprop="title"]')
]

categorie = None

for cat in categories:
    if cat not in categories_generiques:
        categorie = cat
        break

print(categorie)

Jeans


In [56]:
#scrapper pays
get_pays = soup.select_one('[data-testid="seller-location"]')
pays=get_pays.get_text(" ", strip=True) if get_prix else None

pays


'Paris, France'

In [57]:
#scrapper likes
get_likes = soup.select_one('[data-testid="favourite-button"] span.web_ui__Text__text')
likes=get_likes.get_text(" ", strip=True) if get_likes else None

likes

'20'

In [107]:
def scrapping(link):
    response = requests.get(link,headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, 'html.parser')
    infos=["status", "material", "color", "upload_date" ]
    #info
    values=[link]
    for info in infos:
        get_info=soup.select_one(f'[itemprop="{info}"]')
        if get_info:
            info = get_info.get_text(" ", strip=True)
        else:
            info=None
        values.append(info)
    #prix
    get_prix = soup.select_one('[data-testid="item-price"]')
    get_prix_total = soup.select_one('[data-testid="total-combined-price"]')

    prix = get_prix.get_text(" ", strip=True) if get_prix else None
    prix_total=get_prix_total.get_text(" ", strip=True) if get_prix_total else None
    #categories
    categories = [
    tag.get_text(" ", strip=True)
    for tag in soup.select('ul.breadcrumbs span[itemprop="title"]')
    ]

    categorie = None

    for cat in categories:
        if cat not in categories_generiques:
            categorie = cat
            break
    #pays
    get_pays = soup.select_one('[data-testid="seller-location"]')
    if get_pays:
        pays=get_pays.get_text(" ", strip=True)
    else:
        pays=None

    #likes
    get_likes = soup.select_one('[data-testid="favourite-button"] span.web_ui__Text__text')
    if get_likes:
        likes=get_likes.get_text(" ", strip=True)
    else:
        likes=None
            
    values+=[categorie, pays, likes, prix, prix_total]

    return values
    

In [73]:
scrapping(url)

['Très bon état',
 None,
 'Bleu clair',
 'Il y a 3 heures',
 '8,00\xa0€',
 '9,10\xa0€',
 'Jeans',
 'Paris, France',
 '27']

In [108]:
#construction du dataframe
df_vinted_femme=df_vinted.copy()

for i in urls:
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_femme.loc[len(df_vinted_femme)] = ligne
    
df_vinted_femme.head()

https://www.vinted.fr/items/8852156343-robe-dete-zara-taille-s-tres-bon-etat?referrer=catalog
 access to this site is blocked for this computer, due to too many requests


,url,etat,matiere,couleur,date,categorie,pays,likes,prix,prix_total
0,https://www.vinted.fr/items/8851956521-jeans-z...,Très bon état,None,Bleu clair,Il y a 11 heures,Jeans,"Paris, France",41,"8,00 €","9,10 €"
1,https://www.vinted.fr/items/8852816802-jeans-m...,Neuf sans étiquette,None,Bleu clair,Il y a 9 heures,Jeans,None,28,"3,00 €","3,85 €"
2,https://www.vinted.fr/items/8846791444-jean-za...,Très bon état,None,Gris,Il y a 21 heures,Jeans,"Vergèze, France",81,"5,00 €","5,53 €"
3,https://www.vinted.fr/items/8852976048-robe-bu...,Très bon état,None,Rouge,Il y a 8 heures,Robes,None,None,"30,00 €","32,20 €"
4,https://www.vinted.fr/items/8852826952-jogging...,Très bon état,None,Gris,Il y a 9 heures,Vêtements de sport,None,13,"6,00 €","7,00 €"


In [ ]:
df_vinted_femme['collection']="femme"
#df_vinted_femme.to_csv("vinted_data_femme.csv", index=False, encoding="utf-8-sig")



In [114]:
url_homme="https://www.vinted.fr/catalog?catalog[]=5&brand_ids[]=12&page=1&time=1778222504"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup_homme = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [115]:
urls_homme = [link.get('href') for link in soup_homme.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 
print(len(urls_homme))

96


In [116]:
df_vinted_homme=df_vinted.copy()

for i in urls_homme:
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(repr(ligne[-1]), type(ligne[-1]))
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_homme.loc[len(df_vinted_homme)] = ligne
    
df_vinted_homme.head()

None <class 'NoneType'>
https://www.vinted.fr/items/8849470866-salopette-cargo-verde-kaki-zara?referrer=catalog
 access to this site is blocked for this computer, due to too many requests


,url,etat,matiere,couleur,date,categorie,pays,likes,prix,prix_total
0,https://www.vinted.fr/items/8852074529-polo-za...,Neuf sans étiquette,None,Noir,Il y a 11 heures,Hauts et t-shirts,"Saint-Laurent-de-Mure, France",26,"7,00 €","7,70 €"
1,https://www.vinted.fr/items/8853490633-chemise...,Neuf sans étiquette,None,Kaki,Il y a 23 minutes,Hauts et t-shirts,France,5,"12,00 €","13,30 €"
2,https://www.vinted.fr/items/8853408143-pull-bl...,Très bon état,None,Bleu,Il y a une heure,Sweats et pulls,None,2,"2,00 €","2,49 €"
3,https://www.vinted.fr/items/8804537205-pantalo...,Neuf avec étiquette,None,"Crème, Beige",Il y a 9 heures,Pantalons,None,36,"20,00 €","21,70 €"
4,https://www.vinted.fr/items/8849332490-sweat-z...,Neuf sans étiquette,None,Noir,Il y a 16 heures,Sweats et pulls,"Laventie, France",56,"30,00 €","32,20 €"


In [ ]:
df_vinted_homme['collection']="homme"
#df_vinted_homme.to_csv("vinted_data_homme.csv", index=False, encoding="utf-8-sig")

In [121]:
url_fille="https://www.vinted.fr/catalog?brand_ids[]=12&page=1&time=1778223434&catalog[]=1195"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup_fille = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [122]:
urls_fille = [link.get('href') for link in soup_fille.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 

In [123]:
df_vinted_fille=df_vinted.copy()

for i in urls_fille:
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(repr(ligne[-1]), type(ligne[-1]))
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_fille.loc[len(df_vinted_fille)] = ligne

None <class 'NoneType'>
https://www.vinted.fr/items/8850614797-zara-black-drawstring-joggers-size-l?referrer=catalog
 access to this site is blocked for this computer, due to too many requests


In [ ]:
df_vinted_fille['collection']="fille"
#df_vinted_fille.to_csv("vinted_data_fille.csv", index=False, encoding="utf-8-sig")

In [127]:
url_garcon ="https://www.vinted.fr/catalog?brand_ids[]=12&page=1&time=1778224020&catalog[]=1194"
response = requests.get(url,headers={"User-Agent": "Mozilla/5.0"})

# Vérification du statut de la requête
if response.status_code == 200: 
    print("Requête réussie; Code:",response.status_code)
else:
    print("Erreur :", response.status_code)

soup_garcon = BeautifulSoup(response.text, 'html.parser')

Requête réussie; Code: 200


In [129]:
urls_garcon = [link.get('href') for link in soup_garcon.find_all('a',class_=["new-item-box__overlay new-item-box__overlay--clickable" ]) if link.get('href')] 

In [130]:
df_vinted_garcon=df_vinted.copy()

for i in url_garcon:
    ligne=scrapping(i)
    if ligne[-1] is None :
        print(repr(ligne[-1]), type(ligne[-1]))
        print(i)
        print(" access to this site is blocked for this computer, due to too many requests")
        break
    else:
        df_vinted_garcon.loc[len(df_vinted_garcon)] = ligne

None <class 'NoneType'>
https://www.vinted.fr/items/8806259321-jean-flare-zara-s-gris-et-noir?referrer=catalog
 access to this site is blocked for this computer, due to too many requests


In [ ]:
df_vinted_garcon['collection']="garcon"
#df_vinted_garcon.to_csv("vinted_data_garcon.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_total = pd.concat([df_vinted_femme, df_vinted_homme, df_vinted_fille, df_vinted_garcon], ignore_index=True)
#df_total.to_csv("vinted_data.csv", index=False, encoding="utf-8-sig")